**Requerimientos**


In [0]:
%pip install kagglehub[pandas-datasets]
%pip install kaggle
%pip install pyyaml

**Cargando config.yml**

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit
import yaml
import kagglehub
from kagglehub import KaggleDatasetAdapter

with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

**Creando catalogo y esquemas**

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {config['spark']['database']}")
print(f"✅ Catalogo {config['spark']['database']} creado")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['spark']['database']}.bronze")
print(f"✅ Esquema {config['spark']['database']}.bronze creado")

**Descargando datos climáticos** [kaggle](https://www.kaggle.com/datasets/prasad22/weather-data/code)

In [0]:
# Download latest version
path = kagglehub.dataset_download("prasad22/weather-data")

print("Path to dataset files:", path)
print("✅ Dataset descargado")

**Usando spark**
Por la version free community, spark no tiene acceso al tmp ni puede usar DBFS. 
Por lo tando se lee con pandas :C, y luego usamos spark

In [0]:
pdf = pd.read_csv(f"{path}/{config['paths']['raw_data']}")
df = spark.createDataFrame(pdf)
df = df.withColumn("ingestion_date", lit(datetime.today().strftime('%Y-%m-%d')))
display(df.limit(5))
print(f"✅ Dataset cargado en memoria")

**Cargando el archivo crudo a una tabla Delta llamada bronze_weather**

In [0]:
df.write.format("delta") \
    .mode(config["spark"]["write_mode"]) \
    .saveAsTable(config["tables"]["bronze_table"])

print(f"✅ Dataset guardado en {config['tables']['bronze_table']}")
display(spark.sql(f"DESCRIBE DETAIL {config['tables']['bronze_table']}"))